In [1]:
import pandas as pd
import camelot

# 1. pages='all' 會抓取 PDF 內所有偵測到的表格 
path = R"C:\Users\e11338\Desktop\銀泰目錄分割\銀泰螺桿型錄.pdf"
tables = camelot.read_pdf(path, 
                          flavor='lattice', 
                          process_background=True,
                          line_scale=40,
                          pages='all')

print(f"總共偵測到 {len(tables)} 個表格區塊")

# 2. 合併所有表格 
# 注意：上銀型錄每頁的欄位結構可能略有不同（例如 FSV 與 FSI 型），
# 建議先檢查欄位數量是否一致再合併。
all_dfs = [t.df for t in tables]
full_df = pd.concat(all_dfs, ignore_index=True)

# 3. 顯示前 50 行檢查 
display(full_df.head(50))


總共偵測到 157 個表格區塊


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,基本額定負荷(kgf ),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,(1×106 REV.)\nCa(動負荷)Co(靜負荷),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,法蘭,,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A,T,W,G,H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,螺絲孔,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,X,Y,Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,循環圈數,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,圈×列,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [86]:
import pdfplumber
import pandas as pd

path = R"C:\Users\e11338\Desktop\銀泰目錄分割\銀泰螺桿型錄.pdf"

all_tables = []
empt = pd.DataFrame()
with pdfplumber.open(path) as pdf:
    # 遍歷每一頁
    for i, page in enumerate(pdf.pages):
        # 自動偵測表格
        tables = page.extract_tables({
            "vertical_strategy": "text",   # 根據文字對齊判斷縱向欄位 
            "horizontal_strategy": "lines", # 根據線條判斷橫向行 
            "snap_tolerance": 3,           # 容許座標微小誤差
        })
        
        for table in tables:
            df = pd.DataFrame(table)
            # 簡單清理：移除完全為空的列
            df = df.dropna(how='all')
            if not df.empty:
                all_tables.append(df)
                all_tables.append(empt)
        
        print(f"第 {i+1} 頁處理完成，目前抓到 {len(all_tables)} 個表格")

# 合併所有結果
if all_tables:
    page_list = []
    for i in range(len(all_tables)):
        if len(all_tables[i]) > 10:
            page_list.append(all_tables[i])
    print(len(page_list))

    full_df = pd.concat(page_list , ignore_index=True)
    full_df.to_excel("PMI_Extracted_Fixed.xlsx", index=False)
    print("表格擷取成功！已存至 PMI_Extracted_Fixed.xlsx")
else:
    print("依然沒抓到表格，請確認 PDF 是否有鎖加密")

第 1 頁處理完成，目前抓到 4 個表格
第 2 頁處理完成，目前抓到 6 個表格
第 3 頁處理完成，目前抓到 10 個表格
第 4 頁處理完成，目前抓到 12 個表格
第 5 頁處理完成，目前抓到 16 個表格
第 6 頁處理完成，目前抓到 18 個表格
第 7 頁處理完成，目前抓到 22 個表格
第 8 頁處理完成，目前抓到 24 個表格
第 9 頁處理完成，目前抓到 28 個表格
第 10 頁處理完成，目前抓到 30 個表格
第 11 頁處理完成，目前抓到 34 個表格
第 12 頁處理完成，目前抓到 36 個表格
第 13 頁處理完成，目前抓到 40 個表格
第 14 頁處理完成，目前抓到 42 個表格
第 15 頁處理完成，目前抓到 46 個表格
第 16 頁處理完成，目前抓到 48 個表格
第 17 頁處理完成，目前抓到 52 個表格
第 18 頁處理完成，目前抓到 54 個表格
第 19 頁處理完成，目前抓到 58 個表格
第 20 頁處理完成，目前抓到 60 個表格
20
表格擷取成功！已存至 PMI_Extracted_Fixed.xlsx


In [118]:
import pdfplumber
import pandas as pd

path = R"C:\Users\e11338\Desktop\銀泰目錄分割\銀泰螺桿型錄.pdf"

# 定義表格偵測參數
table_settings = {
    "vertical_strategy": "lines",   # 強制改為根據「格線」切欄位，解決文字對不準問題
    "horizontal_strategy": "lines", # 根據橫線切行
    "explicit_vertical_lines": [],  # 如果有漏掉的線可以手動補，先留空
    "explicit_horizontal_lines": [],
    "snap_tolerance": 3,
    "join_tolerance": 3,
}

page_list = []

with pdfplumber.open(path) as pdf:
    for i, page in enumerate(pdf.pages):
        # 取得該頁所有表格
        tables = page.extract_tables(table_settings)
        
        for table in tables:
            df = pd.DataFrame(table)
            
            # --- 關鍵清洗開始 ---
            
            # 1. 移除全空行與全空欄
            df = df.dropna(how='all').dropna(axis=1, how='all')
            
            # 2. 處理儲存格內的換行符 (把 \n 換成空格)
            df = df.replace(r'\n', ' ', regex=True)
            
            # 3. 過濾掉太短的垃圾表格 (通常型錄一頁至少有 5 筆資料)
            if len(df) > 10:
                # 4. 清除「表頭垃圾」的條件過濾
                # 這裡建議用「包含某個關鍵字」來尋找真正的數據起始點
                # 找到包含 "剛性" 或 "油孔" 的最後一行 Index
                mask = df.apply(lambda row: row.astype(str).str.contains("剛性|Q|油孔").any(), axis=1)
                if mask.any():
                    header_idx = df[mask].index[-1]
                    df = df.iloc[header_idx + 1:].reset_index(drop=True).ffill()
                    a = df.iloc[:, :6]
                    b = df.iloc[:,-1]
                    df = pd.concat([a, b], axis = 1)
                                    
                page_list.append(df)
        
        print(f"第 {i+1} 頁處理完成")

# 合併所有 DataFrame
if page_list:
    # 注意：如果各頁欄位數量不同，concat 還是會參差不齊
    # 建議在 concat 前強制統一欄位數量
    # max_cols = max(len(d.columns) for d in page_list)
    # unified_list = [d.iloc[:, :max_cols] for d in page_list] # 截斷多餘欄位或補齊

    full_df = pd.concat(page_list, ignore_index=True)
    full_df.to_excel("PMI_Extracted_Standardized.xlsx", index=False)
    print("標準化表格擷取成功！")
else:
    print("沒抓到任何有效表格")

第 1 頁處理完成
第 2 頁處理完成
第 3 頁處理完成
第 4 頁處理完成
第 5 頁處理完成
第 6 頁處理完成
第 7 頁處理完成
第 8 頁處理完成
第 9 頁處理完成
第 10 頁處理完成
第 11 頁處理完成
第 12 頁處理完成
第 13 頁處理完成
第 14 頁處理完成
第 15 頁處理完成
第 16 頁處理完成
第 17 頁處理完成
第 18 頁處理完成
第 19 頁處理完成
第 20 頁處理完成
標準化表格擷取成功！


In [132]:
import pandas as pd

# 1. 定義規則與欄位名稱
types = ["FSWC", "FDWC", "FSVC", "FDVC", "FOWC"]
groups = [5, 5, 4, 4, 2]
col_name = ["公稱 外徑", "導程", "珠徑", "珠卷數", "動負荷 C (kfg)", "靜負荷 Co (kfg)", "剛性 kfg/umk"]

# 假設 page_list 已經由之前的 pdfplumber 擷取出來
processed_dfs = []
current_idx = 0

# 2. 遍歷系列分組並注入標籤
for i, count in enumerate(groups):
    series_name = types[i]
    for _ in range(count):
        if current_idx < len(page_list):
            df_sub = page_list[current_idx].copy()
            
            # 確保欄位數量對齊
            df_sub = df_sub.iloc[:, :len(col_name)]
            df_sub.columns = col_name
            
            # 注入基礎標籤
            df_sub['brand'] = "PMI"
            df_sub['series'] = series_name
            df_sub['category'] = "Screw"
            df_sub['data_type'] = "Specification" # 標註為規格資料
            
            processed_dfs.append(df_sub)
            current_idx += 1

# 3. 合併總表
if processed_dfs:
    full_df = pd.concat(processed_dfs, ignore_index=True)

    # 4. 格式化「型號」
    def format_model(row):
        try:
            dia = str(row["公稱 外徑"]).strip()
            lead = str(row["導程"]).strip()
            rigidity = str(row["剛性 kfg/umk"]).strip()
            if dia == "" or "nan" in dia.lower() or "None" in dia:
                return "N/A"
            return f"{dia}-{lead}-{rigidity}"
        except:
            return "N/A"

    full_df['model_id'] = full_df.apply(format_model, axis=1)
    full_df = full_df[full_df['model_id'] != "N/A"].reset_index(drop=True)

    # 5. [新增] 產生語意欄位 (Semantic Text)
    def generate_semantic(row):
        return (
            f"這是銀泰 (PMI) 的滾珠螺桿規格。系列名稱為 {row['series']}，"
            f"完整型號為 {row['model_id']}。其主要參數如下：公稱外徑為 {row['公稱 外徑']} mm，"
            f"導程為 {row['導程']} mm，珠徑為 {row['珠徑']} mm，珠卷數為 {row['珠卷數']}。"
            f"在性能指標方面，其動負荷 (Ca) 為 {row['動負荷 C (kfg)']} kgf，"
            f"靜負荷 (Co) 為 {row['靜負荷 Co (kfg)']} kgf，剛性為 {row['剛性 kfg/umk']} kgf/umk。"
        )

    full_df['semantic_text'] = full_df.apply(generate_semantic, axis=1)

    # 6. 最後導出 Excel
    output_file = "PMI_Final_Data_V1.xlsx"
    full_df.to_excel(output_file, index=False)

    print(f"Success: Processed {len(processed_dfs)} tables.")
    print(f"Total rows with semantic text: {len(full_df)}.")
    print(f"File saved to {output_file}.")
else:
    print("Error: page_list is empty.")

Success: Processed 20 tables.
Total rows with semantic text: 590.
File saved to PMI_Final_Data_V1.xlsx.
